# Distributed arrays and halo cells

A tour of `DomainDecomposition` and `DistributedArray`: how a grid is split over MPI ranks, what the ghost (halo) cells hold, and how reductions and operators work. This page runs on a single rank; the same code runs unchanged under `mpiexec -n N`.

In [ ]:
import cunumpy as xp
import numpy as np

from mpiarray import DistributedArray, DomainDecomposition, calculate_proc_sizes

# mpi4py.MPI under mpiexec, otherwise cunumpy's serial stand-in
MPI = xp.mpi.get_mpi()
comm = MPI.COMM_WORLD
print(comm, "with", comm.size, "rank(s)")

## Splitting a grid

`calculate_proc_sizes` picks the process grid for a number of ranks, splitting only the axes flagged for decomposition. It needs no MPI, so we can look at layouts for more ranks than this page runs on:

In [ ]:
for size in (4, 6, 12, 7):
    print(size, "ranks ->", calculate_proc_sizes(size, [True, True, False]))

A prime number of ranks cannot be split over two axes, so it all goes to the first one.

`DomainDecomposition` applies this to a communicator and also gives each rank its neighbours and the index range it owns:

In [ ]:
layout = DomainDecomposition(comm, decompose=[True, True], periodic=(True, False))
print("process grid:", layout.proc_sizes)
print("neighbours (left, right) per axis:", layout.neighbour_ranks)
print("owned index ranges of a 64 x 48 grid:", layout.get_index_bounds((64, 48)))

On one rank, the periodic axis has this rank as its own neighbour, and the wall axis has `MPI.PROC_NULL` on both sides.

## Ghost cells

A `DistributedArray` stores each rank's block with `num_ghostpoints` extra cells on each side. `local` is the interior, `local_with_halos` includes the ghost cells:

In [ ]:
a = DistributedArray.from_array(
    np.arange(8.0), comm=comm, num_ghostpoints=1, periodic=(True,)
)
print("interior:   ", a.local)
print("with halos: ", a.local_with_halos)

`fill_halos()` copies the neighbours' boundary values into the ghost cells, as a finite-difference stencil needs. On this periodic axis the ghosts wrap around:

In [ ]:
a.fill_halos()
print(a.local_with_halos)

# a centred difference only needs the local block with its halos
u = a.local_with_halos
print("second difference:", u[:-2] - 2 * u[1:-1] + u[2:])

`exchange_halos()` goes the other way: values written into ghost cells, for example charge deposited by particles near a subdomain edge, are added into the neighbour's interior, and the ghosts are zeroed:

In [ ]:
rho = DistributedArray.zeros(shape=(8,), comm=comm, num_ghostpoints=1, periodic=(True,))
rho.local_with_halos[0] = 1.0  # deposited left of the first interior cell
rho.local_with_halos[-1] = 2.0  # deposited right of the last one
rho.exchange_halos()
print(rho.local_with_halos)

## Reductions and operators

Reductions are global (collective over the communicator) and ignore ghost cells. Arithmetic and NumPy ufuncs return new distributed arrays with the same layout:

In [ ]:
b = DistributedArray.from_array(np.linspace(0.0, 1.0, 8), comm=comm)
c = 2 * b + np.sin(b)
print("sum:", c.sum(), " max:", c.max(), " norm:", c.norm())
print("gathered on every rank:", c.to_ndarray())

## On several ranks

Save the cells above as a script and run it with `mpiexec -n 4 python tutorial.py`: each rank then holds its own block, `fill_halos` and `exchange_halos` exchange messages with the neighbour ranks, and `sum`, `norm` and `to_ndarray` communicate. On the CuPy backend (`CUNUMPY_BACKEND=cupy`), call `xp.mpi.set_mpi_cuda_aware(...)` or `xp.mpi.mpi_is_cuda_aware(comm)` once first.